In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the three datasets
customers = pd.read_csv("../Customers_list.csv")
products = pd.read_csv("../Products_list.csv")
transactions = pd.read_csv("../Transactions_list.csv")

# Display dataset sizes
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Transactions:", transactions.shape)

# Display the first five rows of each dataset
display(customers.head())
display(products.head())
display(transactions.head())


Customers: (1000, 5)
Products: (200, 6)
Transactions: (18829, 5)


,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,50+,South,Bronze,8
1,2,26-35,South,Bronze,1
2,3,26-35,West,Bronze,2
3,4,50+,West,Silver,2
4,5,26-35,South,Bronze,8


,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Fitness,PrimeGoods,66.68,4.35,1
1,2,Kitchen,FitLife,367.01,NaN,2
2,3,Books,TechMax,176.00,4.57,1
3,4,Fitness,NextGen,426.37,3.39,7
4,5,Gaming,TechMax,266.57,4.19,1


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,171,1,82,462.89
1,2,48,4,255,433.60
2,2,112,3,315,1264.65
3,3,1,1,53,460.84
4,4,44,4,269,767.68


pd.read_csv() loads your CSV files into Pandas DataFrames.



.shape tells us how many rows and columns each dataset contains.



.head() displays the first five rows so we can understand the data structure.

In [2]:

# Check missing values in all three datasets

print("Customer Missing Values:")
print(customers.isnull().sum())

print("\nProduct Missing Values:")
print(products.isnull().sum())

print("\nTransaction Missing Values:")
print(transactions.isnull().sum())


Customer Missing Values:
CustomerID         0
AgeGroup           0
Region             0
LoyaltyLevel       0
MembershipYears    0
dtype: int64

Product Missing Values:
ProductID           0
Category            0
Brand               0
Price               0
Rating             15
PopularityScore     0
dtype: int64

Transaction Missing Values:
CustomerID          0
ProductID           0
PurchaseCount       0
LastPurchaseDays    0
Spend               0
dtype: int64


.isnull() checks whether a value is missing.



.sum() counts the missing values in each column.



print() displays the results separately for each dataset.

In [3]:

# Check duplicate rows in each dataset

print("Customer duplicates:",
      customers.duplicated().sum())

print("Product duplicates:",
      products.duplicated().sum())

print("Transaction duplicates:",
      transactions.duplicated().sum())

# Check duplicate customer and product IDs
print("\nDuplicate Customer IDs:",
      customers["CustomerID"].duplicated().sum())

print("Duplicate Product IDs:",
      products["ProductID"].duplicated().sum())


Customer duplicates: 0
Product duplicates: 0
Transaction duplicates: 386

Duplicate Customer IDs: 0
Duplicate Product IDs: 0


In [4]:

# Display examples of duplicate transactions
duplicate_transactions = transactions[
    transactions.duplicated(keep=False)
]

print("Examples of duplicate transactions:")
display(duplicate_transactions.head(10))

# Check how many unique customer-product pairs exist
print("Unique customer-product pairs:")
print(
    transactions[
        ["CustomerID", "ProductID"]
    ].drop_duplicates().shape[0]
)


Examples of duplicate transactions:


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
4,4,44,4,269,767.68
22,17,27,3,147,895.41
108,53,186,2,303,641.16
131,54,161,1,8,298.26
134,54,72,4,224,1369.48
142,54,161,1,8,298.26
153,55,64,2,78,332.48
201,57,101,3,105,1471.32
205,57,55,2,278,33.04
213,58,166,4,196,1573.32


Unique customer-product pairs:
10263


This lets us inspect whether the duplicate records appear to be repeated entries.

In [5]:

# Remove exact duplicate transaction rows
transactions_clean = transactions.drop_duplicates().copy()

print("Original transaction records:", len(transactions))
print("Cleaned transaction records:", len(transactions_clean))
print("Duplicates remaining:",
      transactions_clean.duplicated().sum())


Original transaction records: 18829
Cleaned transaction records: 18443
Duplicates remaining: 0


In [6]:

# Remove exact duplicate transaction rows
transactions_clean = transactions.drop_duplicates().copy()

print("Original transaction records:", len(transactions))
print("Cleaned transaction records:", len(transactions_clean))
print("Duplicates remaining:",
      transactions_clean.duplicated().sum())


Original transaction records: 18829
Cleaned transaction records: 18443
Duplicates remaining: 0


In [7]:

# Create a copy of the products dataset
products_clean = products.copy()

# Calculate median product rating
median_rating = products_clean["Rating"].median()

# Replace missing ratings with median
products_clean["Rating"] = products_clean[
    "Rating"
].fillna(median_rating)

# Check remaining missing values
print("Median rating:", median_rating)

print("\nMissing values after cleaning:")
print(products_clean.isnull().sum())


Median rating: 3.87

Missing values after cleaning:
ProductID          0
Category           0
Brand              0
Price              0
Rating             0
PopularityScore    0
dtype: int64


In [8]:

# Remove exact duplicate transactions
transactions_clean = transactions.drop_duplicates().copy()

print("Original transactions:", len(transactions))
print("Cleaned transactions:", len(transactions_clean))
print("Duplicates remaining:",
      transactions_clean.duplicated().sum())


Original transactions: 18829
Cleaned transactions: 18443
Duplicates remaining: 0


In [9]:

# Combine transactions with customer information
combined_data = transactions_clean.merge(
    customers,
    on="CustomerID",
    how="left",
    validate="many_to_one"
)

# Add product information
combined_data = combined_data.merge(
    products_clean,
    on="ProductID",
    how="left",
    validate="many_to_one"
)

print("Combined dataset shape:", combined_data.shape)

# Display first five rows
display(combined_data.head())

# Check whether customer or product records failed to match
print("Missing customer matches:",
      combined_data["AgeGroup"].isnull().sum())

print("Missing product matches:",
      combined_data["Category"].isnull().sum())


Combined dataset shape: (18443, 14)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend,AgeGroup,Region,LoyaltyLevel,MembershipYears,Category,Brand,Price,Rating,PopularityScore
0,1,171,1,82,462.89,50+,South,Bronze,8,Electronics,TechMax,449.12,4.36,2
1,2,48,4,255,433.60,26-35,South,Bronze,1,Gaming,FitLife,273.31,3.90,8
2,2,112,3,315,1264.65,26-35,South,Bronze,1,Home,NextGen,205.16,3.07,15
3,3,1,1,53,460.84,26-35,West,Bronze,2,Fitness,PrimeGoods,66.68,4.35,1
4,4,44,4,269,767.68,50+,West,Silver,2,Gaming,TechMax,136.66,4.72,1


Missing customer matches: 0
Missing product matches: 0


In [10]:

# Combine transactions with customer information
combined_data = transactions_clean.merge(
    customers,
    on="CustomerID",
    how="left",
    validate="many_to_one"
)

# Add product information
combined_data = combined_data.merge(
    products_clean,
    on="ProductID",
    how="left",
    validate="many_to_one"
)

print("Combined dataset shape:", combined_data.shape)

# Display first five rows
display(combined_data.head())

# Check whether customer or product records failed to match
print("Missing customer matches:",
      combined_data["AgeGroup"].isnull().sum())

print("Missing product matches:",
      combined_data["Category"].isnull().sum())


Combined dataset shape: (18443, 14)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend,AgeGroup,Region,LoyaltyLevel,MembershipYears,Category,Brand,Price,Rating,PopularityScore
0,1,171,1,82,462.89,50+,South,Bronze,8,Electronics,TechMax,449.12,4.36,2
1,2,48,4,255,433.60,26-35,South,Bronze,1,Gaming,FitLife,273.31,3.90,8
2,2,112,3,315,1264.65,26-35,South,Bronze,1,Home,NextGen,205.16,3.07,15
3,3,1,1,53,460.84,26-35,West,Bronze,2,Fitness,PrimeGoods,66.68,4.35,1
4,4,44,4,269,767.68,50+,West,Silver,2,Gaming,TechMax,136.66,4.72,1


Missing customer matches: 0
Missing product matches: 0


In [11]:

# Create customer-product interaction matrix
interaction_matrix = combined_data.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

print("Interaction Matrix Shape:")
print(interaction_matrix.shape)

# Display first five customers and ten products
display(interaction_matrix.iloc[:5, :10])


Interaction Matrix Shape:
(1000, 200)


ProductID,1,2,3,4,5,6,7,8,9,10
CustomerID,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,2,0,0,0,0


In [12]:

# Count customer-product combinations with purchases
total_interactions = (interaction_matrix > 0).sum().sum()

# Calculate total possible combinations
possible_interactions = interaction_matrix.size

# Calculate matrix density
density = (
    total_interactions / possible_interactions
) * 100

print("Recorded customer-product interactions:",
      total_interactions)

print("Possible customer-product interactions:",
      possible_interactions)

print(f"Interaction matrix density: {density:.2f}%")
print(f"Interaction matrix sparsity: {100 - density:.2f}%")


Recorded customer-product interactions: 10263
Possible customer-product interactions: 200000
Interaction matrix density: 5.13%
Interaction matrix sparsity: 94.87%


In [13]:

# Calculate total spending for each customer
customer_spending = combined_data.groupby(
    "CustomerID"
)["Spend"].sum().sort_values(ascending=False)

# Select the top 10 customers
top_10_spending = customer_spending.head(10)

print("Top 10 Customers by Total Spending:")
display(top_10_spending)


Top 10 Customers by Total Spending:


CustomerID
729    23828.03
660    23301.22
894    23127.86
481    22819.08
74     22564.03
83     21968.20
707    21746.12
544    21681.06
576    21668.43
818    21549.61
Name: Spend, dtype: float64

In [14]:

# Calculate total purchases for each customer
customer_frequency = combined_data.groupby(
    "CustomerID"
)["PurchaseCount"].sum().sort_values(ascending=False)

# Select the top 10 customers
top_10_frequency = customer_frequency.head(10)

print("Top 10 Customers by Purchase Frequency:")
display(top_10_frequency)


Top 10 Customers by Purchase Frequency:


CustomerID
450    83
374    82
298    82
74     82
880    82
329    81
240    81
168    80
65     79
681    78
Name: PurchaseCount, dtype: int64

groupby("CustomerID") groups records by customer.



["PurchaseCount"].sum() calculates the total number of purchases recorded for each customer.



sort_values(ascending=False) orders customers from most purchases to fewest.



.head(10) selects the 10 most frequent purchasers.

In [15]:

# Analyse customer value by loyalty level
customer_value = combined_data.groupby(
    "LoyaltyLevel"
).agg(
    TotalRevenue=("Spend", "sum"),
    TotalPurchases=("PurchaseCount", "sum"),
    UniqueCustomers=("CustomerID", "nunique")
)

# Calculate average spending per customer
customer_value["AverageSpendPerCustomer"] = (
    customer_value["TotalRevenue"] /
    customer_value["UniqueCustomers"]
)

# Calculate average purchases per customer
customer_value["AveragePurchasesPerCustomer"] = (
    customer_value["TotalPurchases"] /
    customer_value["UniqueCustomers"]
)

print("Customer Value by Loyalty Level:")
display(customer_value.round(2))


Customer Value by Loyalty Level:


,TotalRevenue,TotalPurchases,UniqueCustomers,AverageSpendPerCustomer,AveragePurchasesPerCustomer
LoyaltyLevel,,,,,
Bronze,5434949.05,22461,493,11024.24,45.56
Gold,2101572.03,8645,187,11238.35,46.23
Silver,3562339.29,14846,320,11132.31,46.39


In [16]:

# Calculate total revenue for each product
product_revenue = combined_data.groupby(
    "ProductID"
).agg(
    TotalRevenue=("Spend", "sum"),
    TotalPurchases=("PurchaseCount", "sum"),
    Category=("Category", "first"),
    Brand=("Brand", "first")
)

# Sort products by revenue
top_10_products = product_revenue.sort_values(
    by="TotalRevenue",
    ascending=False
).head(10)

print("Top 10 Products by Total Revenue:")
display(top_10_products.round(2))


Top 10 Products by Total Revenue:


,TotalRevenue,TotalPurchases,Category,Brand
ProductID,,,,
64,969511.68,5832,Books,PrimeGoods
76,516181.63,2383,Kitchen,FitLife
73,513764.73,1041,Gaming,TechMax
137,377406.54,1447,Fitness,NextGen
18,285252.63,601,Gaming,SmartBuy
47,285250.14,621,gaming,NextGen
12,270583.58,2326,fitness,PrimeGoods
6,207772.24,664,Kitchen,SmartBuy
44,201899.84,1052,Gaming,TechMax


In [17]:

# Standardise category names
combined_data["Category"] = (
    combined_data["Category"]
    .str.strip()
    .str.title()
)

# Standardise category names in the cleaned products dataset
products_clean["Category"] = (
    products_clean["Category"]
    .str.strip()
    .str.title()
)

# Calculate revenue and purchases by category
category_revenue = combined_data.groupby(
    "Category"
).agg(
    TotalRevenue=("Spend", "sum"),
    TotalPurchases=("PurchaseCount", "sum"),
    UniqueProducts=("ProductID", "nunique")
).sort_values(
    by="TotalRevenue",
    ascending=False
)

print("Revenue by Product Category:")
display(category_revenue.round(2))


Revenue by Product Category:


,TotalRevenue,TotalPurchases,UniqueProducts
Category,,,
Books,2335204.47,10847,35
Gaming,2281265.76,7327,27
Kitchen,1868036.68,7163,35
Fitness,1826611.28,8895,38
Electronics,1445638.65,5571,32
Home,1342103.53,6149,33


.str.strip() removes unnecessary spaces from category names.

.str.title() standardises capitalisation.

groupby("Category") groups transactions by product category.

TotalRevenue calculates how much revenue each category generates.

TotalPurchases calculates the number of purchases.

UniqueProducts counts the different products in each category.

In [18]:

# Standardise brand names
combined_data["Brand"] = (
    combined_data["Brand"]
    .str.strip()
    .str.title()
)

products_clean["Brand"] = (
    products_clean["Brand"]
    .str.strip()
    .str.title()
)

# Calculate revenue by brand
brand_revenue = combined_data.groupby(
    "Brand"
).agg(
    TotalRevenue=("Spend", "sum"),
    TotalPurchases=("PurchaseCount", "sum"),
    UniqueProducts=("ProductID", "nunique")
).sort_values(
    by="TotalRevenue",
    ascending=False
)

print("Revenue by Brand:")
display(brand_revenue.round(2))


Revenue by Brand:


,TotalRevenue,TotalPurchases,UniqueProducts
Brand,,,
Techmax,2615417.61,10240,45
Primegoods,2454682.07,13997,33
Fitlife,1712054.34,6656,34
Nextgen,1536967.99,5557,27
Smartbuy,1420657.50,4874,29
Homepro,1359080.86,4628,32


.str.strip() removes unnecessary spaces.

.str.title() standardises brand names so inconsistent capitalisation doesn't split the same brand into separate groups.

groupby("Brand") groups the transaction data by brand.

TotalRevenue calculates the revenue associated with each brand.

TotalPurchases calculates the number of purchases.

UniqueProducts counts how many different products belong to each brand.

In [19]:

# Pivot Table 1: Revenue by region and loyalty level
pivot_region_loyalty = combined_data.pivot_table(
    index="Region",
    columns="LoyaltyLevel",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

print("Pivot Table 1: Revenue by Region and Loyalty Level")
display(pivot_region_loyalty.round(2))


Pivot Table 1: Revenue by Region and Loyalty Level


LoyaltyLevel,Bronze,Gold,Silver
Region,,,
East,1195197.64,581170.24,793626.51
North,1362121.31,409313.47,974312.01
South,1518311.01,498256.42,922215.35
West,1359319.09,612831.90,872185.42


index="Region" places regions in the rows.

columns="LoyaltyLevel" places Bronze, Silver and Gold in the columns.

values="Spend" selects customer spending.

aggfunc="sum" calculates total revenue for each combination.

In [20]:

# Pivot Table 2: Purchases by category and loyalty level
pivot_category_loyalty = combined_data.pivot_table(
    index="Category",
    columns="LoyaltyLevel",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

print("Purchases by Category and Loyalty Level:")
display(pivot_category_loyalty)


Purchases by Category and Loyalty Level:


LoyaltyLevel,Bronze,Gold,Silver
Category,,,
Books,5437,1837,3573
Electronics,2566,1109,1896
Fitness,4412,1624,2859
Gaming,3622,1408,2297
Home,2889,1350,1910
Kitchen,3535,1317,2311


In [21]:

# Calculate product popularity based on purchases
popular_products = combined_data.groupby(
    "ProductID"
).agg(
    TotalPurchases=("PurchaseCount", "sum"),
    TotalRevenue=("Spend", "sum"),
    Category=("Category", "first"),
    Brand=("Brand", "first")
)

# Rank products by purchase count
popular_products = popular_products.sort_values(
    by="TotalPurchases",
    ascending=False
)

# Display top 10 popular products
print("Top 10 Most Popular Products:")
display(popular_products.head(10))


Top 10 Most Popular Products:


,TotalPurchases,TotalRevenue,Category,Brand
ProductID,,,,
64,5832,969511.68,Books,Primegoods
76,2383,516181.63,Kitchen,Fitlife
12,2326,270583.58,Fitness,Primegoods
137,1447,377406.54,Fitness,Nextgen
198,1405,74493.10,Fitness,Primegoods
44,1052,201899.84,Gaming,Techmax
73,1041,513764.73,Gaming,Techmax
114,873,126873.09,Home,Primegoods
6,664,207772.24,Kitchen,Smartbuy


In [22]:

# Function to recommend popular products
def recommend_popular_products(top_n=5):
    return popular_products.head(top_n)

# Generate five recommendations
recommendations = recommend_popular_products(5)

print("Top 5 Popular Product Recommendations:")
display(recommendations)


Top 5 Popular Product Recommendations:


,TotalPurchases,TotalRevenue,Category,Brand
ProductID,,,,
64,5832,969511.68,Books,Primegoods
76,2383,516181.63,Kitchen,Fitlife
12,2326,270583.58,Fitness,Primegoods
137,1447,377406.54,Fitness,Nextgen
198,1405,74493.10,Fitness,Primegoods


In [23]:

# Recommend popular products a customer hasn't purchased
def recommend_popular_for_customer(customer_id, top_n=5):

    # Find products the customer has already purchased
    purchased_products = combined_data.loc[
        combined_data["CustomerID"] == customer_id,
        "ProductID"
    ].unique()

    # Remove purchased products from recommendations
    available_products = popular_products.drop(
        index=purchased_products,
        errors="ignore"
    )

    # Return the most popular remaining products
    return available_products.head(top_n)


# Test the function with Customer 1
customer_id = 1

print(f"Top 5 Popular Recommendations for Customer {customer_id}:")
display(recommend_popular_for_customer(customer_id))


Top 5 Popular Recommendations for Customer 1:


,TotalPurchases,TotalRevenue,Category,Brand
ProductID,,,,
64,5832,969511.68,Books,Primegoods
76,2383,516181.63,Kitchen,Fitlife
12,2326,270583.58,Fitness,Primegoods
137,1447,377406.54,Fitness,Nextgen
198,1405,74493.10,Fitness,Primegoods


In [24]:

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Calculate similarity between customers
customer_similarity = cosine_similarity(interaction_matrix)

# Convert similarity scores into a DataFrame
customer_similarity_df = pd.DataFrame(
    customer_similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

print("Customer Similarity Matrix Shape:")
print(customer_similarity_df.shape)

print("\nSimilarity Scores for the First 5 Customers:")
display(customer_similarity_df.iloc[:5, :5].round(3))


Customer Similarity Matrix Shape:
(1000, 1000)

Similarity Scores for the First 5 Customers:


CustomerID,1,2,3,4,5
CustomerID,,,,,
1,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0
5,0.0,0.0,0.0,0.0,1.0


cosine_similarity() compares customers based on their product purchase patterns.

Customers who purchase similar products receive higher similarity scores.

pd.DataFrame() converts the results into a table.

.iloc[:5, :5] displays similarity scores for the first five customers.

In [25]:

# Select the customer to analyse
customer_id = 1

# Get similarity scores for Customer 1
similar_customers = customer_similarity_df.loc[
    customer_id
].drop(customer_id)

# Sort customers by similarity
similar_customers = similar_customers.sort_values(
    ascending=False
)

# Display the five most similar customers
print(f"Top 5 Customers Most Similar to Customer {customer_id}:")
display(similar_customers.head(5).round(3))


Top 5 Customers Most Similar to Customer 1:


CustomerID
191    0.371
183    0.343
251    0.340
729    0.312
605    0.249
Name: 1, dtype: float64

.loc[customer_id] retrieves similarity scores for Customer 1.



.drop(customer_id) removes Customer 1 from the results because customers shouldn't be compared with themselves when generating recommendations.



.sort_values(ascending=False) ranks the remaining customers from most similar to least similar.



.head(5) displays the five most similar customers.

In [26]:

# User-based collaborative filtering
def recommend_collaborative(customer_id, top_n=5, neighbours=20):

    # Find similar customers, excluding the target customer
    similar_users = (
        customer_similarity_df.loc[customer_id]
        .drop(customer_id)
        .sort_values(ascending=False)
        .head(neighbours)
    )

    # Keep only customers with positive similarity
    similar_users = similar_users[similar_users > 0]

    # Get purchase counts for these customers
    neighbour_purchases = interaction_matrix.loc[
        similar_users.index
    ]

    # Calculate similarity-weighted recommendation scores
    scores = neighbour_purchases.T.dot(similar_users)

    # Exclude products already purchased by the customer
    purchased_products = interaction_matrix.loc[customer_id]
    purchased_products = purchased_products[
        purchased_products > 0
    ].index

    scores = scores.drop(
        index=purchased_products,
        errors="ignore"
    )

    # Remove products with no recommendation score
    scores = scores[scores > 0]

    # Return the highest-scoring products
    return scores.sort_values(
        ascending=False
    ).head(top_n)


# Test collaborative filtering with Customer 1
customer_id = 1

print(f"Collaborative Recommendations for Customer {customer_id}:")
display(recommend_collaborative(customer_id))


Collaborative Recommendations for Customer 1:


ProductID
44     27.734231
114    27.373394
47     15.278286
101    14.930383
19     13.713710
dtype: float64

In [27]:

# Select five customers for testing
example_customers = [1, 2, 3, 4, 5]

# Store results for comparison
comparison_results = []

for customer_id in example_customers:

    # Generate popularity-based recommendations
    popularity_recs = recommend_popular_for_customer(
        customer_id,
        top_n=5
    )

    # Generate collaborative filtering recommendations
    collaborative_recs = recommend_collaborative(
        customer_id,
        top_n=5
    )

    # Store the recommended product IDs
    comparison_results.append({
        "CustomerID": customer_id,
        "PopularityRecommendations": popularity_recs.index.tolist(),
        "CollaborativeRecommendations": collaborative_recs.index.tolist()
    })

# Convert results into a DataFrame
comparison_df = pd.DataFrame(comparison_results)

print("Recommendation System Comparison:")
display(comparison_df)


Recommendation System Comparison:


,CustomerID,PopularityRecommendations,CollaborativeRecommendations
0,1,"[64, 76, 12, 137, 198]","[44, 114, 47, 101, 19]"
1,2,"[64, 76, 12, 137, 198]","[137, 198, 73, 8, 155]"
2,3,"[64, 76, 12, 137, 198]","[12, 6, 75, 110, 56]"
3,4,"[64, 76, 12, 137, 198]","[114, 19, 47, 101, 187]"
4,5,"[64, 76, 12, 137, 198]","[12, 75, 110, 1, 164]"


Selects five customers.

Runs the popularity-based algorithm for each customer.

Runs collaborative filtering for each customer.

Stores the recommended product IDs.

Displays both sets of recommendations side by side.

In [28]:

from sklearn.model_selection import train_test_split

# Get unique customer-product interactions
interactions = combined_data[
    ["CustomerID", "ProductID", "PurchaseCount"]
].groupby(
    ["CustomerID", "ProductID"],
    as_index=False
)["PurchaseCount"].sum()

# Split each eligible customer's interactions
# into training and testing sets
train_parts = []
test_parts = []

for customer_id, group in interactions.groupby("CustomerID"):

    if len(group) < 2:
        train_parts.append(group)
        continue

    train_group, test_group = train_test_split(
        group,
        test_size=0.2,
        random_state=42
    )

    train_parts.append(train_group)
    test_parts.append(test_group)

train_interactions = pd.concat(
    train_parts,
    ignore_index=True
)

test_interactions = pd.concat(
    test_parts,
    ignore_index=True
)

print("Training interactions:", len(train_interactions))
print("Testing interactions:", len(test_interactions))

print(
    "Customers in testing set:",
    test_interactions["CustomerID"].nunique()
)


Training interactions: 7839
Testing interactions: 2424
Customers in testing set: 966


Combines repeated purchases of the same product by the same customer.

Splits each eligible customer's purchased products into training and testing sets.

Keeps approximately 80% of each customer's product interactions for training.

Holds out approximately 20% for testing.

Keeps customers with only one interaction in training, because they cannot be split into both sets.

In [29]:

# Create training customer-product interaction matrix
train_matrix = train_interactions.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

# Include all original customers and products
train_matrix = train_matrix.reindex(
    index=interaction_matrix.index,
    columns=interaction_matrix.columns,
    fill_value=0
)

# Rebuild popularity ranking using training data only
train_popularity = train_interactions.groupby(
    "ProductID"
)["PurchaseCount"].sum().sort_values(
    ascending=False
)

# Rebuild customer similarity using training data only
train_similarity = cosine_similarity(train_matrix)

train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

print("Training Interaction Matrix:", train_matrix.shape)
print("Training Similarity Matrix:", train_similarity_df.shape)

print("\nTop 5 Training Popular Products:")
display(train_popularity.head(5))


Training Interaction Matrix: (1000, 200)
Training Similarity Matrix: (1000, 1000)

Top 5 Training Popular Products:


ProductID
64     3055
76     2148
12     1764
198    1193
137    1167
Name: PurchaseCount, dtype: int64

train_matrix creates a new customer-product matrix containing only training purchases.

reindex() ensures the matrix includes the same customers and products as the original.

train_popularity recalculates product popularity without using testing purchases.

cosine_similarity() recalculates customer similarity based only on training data.

train_similarity_df stores the similarity scores for our evaluation.

In [30]:

# Popularity recommendations using training data only
def evaluate_popularity(customer_id, top_n=5):

    # Products already purchased in training
    purchased = train_matrix.loc[customer_id]
    purchased = purchased[purchased > 0].index

    # Exclude previously purchased products
    recommendations = train_popularity.drop(
        index=purchased,
        errors="ignore"
    )

    return recommendations.head(top_n).index.tolist()


# Collaborative recommendations using training data only
def evaluate_collaborative(customer_id, top_n=5, neighbours=20):

    # Find similar customers
    similar_users = (
        train_similarity_df.loc[customer_id]
        .drop(customer_id)
        .sort_values(ascending=False)
        .head(neighbours)
    )

    similar_users = similar_users[similar_users > 0]

    # Get purchases made by similar customers
    neighbour_purchases = train_matrix.loc[
        similar_users.index
    ]

    # Calculate weighted recommendation scores
    scores = neighbour_purchases.T.dot(similar_users)

    # Exclude products already purchased in training
    purchased = train_matrix.loc[customer_id]
    purchased = purchased[purchased > 0].index

    scores = scores.drop(
        index=purchased,
        errors="ignore"
    )

    # Keep products with positive scores
    scores = scores[scores > 0]

    return scores.sort_values(
        ascending=False
    ).head(top_n).index.tolist()


# Test both functions with Customer 1
print("Popularity Recommendations:")
print(evaluate_popularity(1))

print("\nCollaborative Recommendations:")
print(evaluate_collaborative(1))


Popularity Recommendations:
[64, 76, 12, 198, 137]

Collaborative Recommendations:
[114, 44, 47, 134, 192]


In [31]:

# Calculate Precision@5
def precision_at_5(recommendations, actual_products):
    relevant = len(
        set(recommendations) & set(actual_products)
    )

    return relevant / 5


# Store evaluation results
evaluation_results = []

# Evaluate each customer in the testing dataset
for customer_id, group in test_interactions.groupby("CustomerID"):

    # Products held out for testing
    actual_products = group["ProductID"].tolist()

    # Generate recommendations
    popularity_recs = evaluate_popularity(customer_id)
    collaborative_recs = evaluate_collaborative(customer_id)

    # Calculate precision
    popularity_precision = precision_at_5(
        popularity_recs,
        actual_products
    )

    collaborative_precision = precision_at_5(
        collaborative_recs,
        actual_products
    )

    evaluation_results.append({
        "CustomerID": customer_id,
        "PopularityPrecision": popularity_precision,
        "CollaborativePrecision": collaborative_precision
    })


# Convert results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_results)

# Calculate average Precision@5
average_precision = evaluation_df[
    ["PopularityPrecision", "CollaborativePrecision"]
].mean()

print("Average Precision@5:")
display(average_precision.round(4))

print("\nNumber of Customers Evaluated:")
print(len(evaluation_df))


Average Precision@5:


PopularityPrecision       0.0460
CollaborativePrecision    0.1542
dtype: float64


Number of Customers Evaluated:
966


precision_at_5() checks how many recommended products appear in the customer's testing purchases.

The loop evaluates both algorithms for every customer in the testing dataset.

Each customer's Precision@5 score is stored in evaluation_results.

evaluation_df converts the results into a DataFrame.

.mean() calculates the average Precision@5 for each algorithm.

In [32]:

# Compare recommendation algorithm performance
performance_comparison = pd.DataFrame({
    "Algorithm": [
        "Popularity-Based",
        "Collaborative Filtering"
    ],
    "Precision@5": [
        evaluation_df["PopularityPrecision"].mean(),
        evaluation_df["CollaborativePrecision"].mean()
    ]
})

performance_comparison["Precision (%)"] = (
    performance_comparison["Precision@5"] * 100
)

performance_comparison = performance_comparison.sort_values(
    by="Precision@5",
    ascending=False
)

print("Recommendation Algorithm Performance Comparison:")
display(performance_comparison.round(4))


Recommendation Algorithm Performance Comparison:


,Algorithm,Precision@5,Precision (%)
1,Collaborative Filtering,0.1542,15.4244
0,Popularity-Based,0.0460,4.5963


Precision5
Precision@5 measures how many of the five products recommended by an algorithm match products the customer actually purchased in the testing dataset. The percentages show that on what average a recommended product matches products the customers purchased in the testing dataset.

Popularity-based recommendation system
The algorithm we used was able to achieve a precision5 result of 4.6%. the algorithm recommend products based on their total purchase counts across all customers the main stengths of using it included the little use of computation power, the ability to recommend products to new customerts without a purchase history as well as providing us with a useful baseline for evaluating other algorithims. Despite the benefits it did come with its fair share of drawbacks such as its tendenancy to often recommend the same product to customers as well as leading to less popular products recieving less exposure than others.

Collaborative filtering recommendation system
The user based collaborative filtering algorithm was given a score of 15.42%. The collaborative filtering recommendation system used to identifiy cutsomers who had similar purchasing patterns using the cosine similarity, the main strenghts of this were that recommendations became more personalised, a draw back of this was that the user needs a large purchase history for it to feel personalised enough. This algorithm also allowde for products that werent globally popular to reach new audiances as well as being able to identify patterns between customers despite calculating similarities becoming more computationally expensive as the number of customers grow.

Results
The collaborative filtering system used information about customers with similar purchase histories and behaviours, in comparison the popularity based system relied on overall purchase frequency, this information appeared to have helped collaborative filtering identify products that matched cutsomers held-out purchases more frequently, despite this both systems both had their limitations.